# Astro Hunter — 01: prima light curve TESS
Obiettivo: scaricare dati reali di **Pi Mensae (TIC 261136679)** dal Sector 1 di TESS e capire che cosa contiene una light curve.


In [ ]:
import lightkurve as lk
import matplotlib.pyplot as plt


## 1. Cerca il dato in MAST
`search_lightcurve` interroga l'archivio pubblico MAST. Limitiamo la ricerca ai prodotti SPOC da 120 s del Sector 1.


In [ ]:
target = 'TIC 261136679'
sector = 1
search = lk.search_lightcurve(target, mission='TESS', sector=sector, author='SPOC', exptime=120)
search


## 2. Scarica e ispeziona
Usiamo la maschera di qualità standard di TESS. Se disponibile, scegliamo esplicitamente `pdcsap_flux`, cioè la fotometria corretta dalla pipeline SPOC per molte sistematiche strumentali.


In [ ]:
lc = search.download(quality_bitmask='default')
lc.colnames


In [ ]:
if 'pdcsap_flux' in lc.colnames:
    lc = lc.select_flux('pdcsap_flux')
lc


## 3. Pulizia minima
Rimuoviamo NaN, normalizziamo il flusso alla mediana (=1) e togliamo outlier estremi oltre 6σ. Non stiamo ancora cercando il pianeta.


In [ ]:
clean = lc.remove_nans().normalize().remove_outliers(sigma=6)
print(f'Cadences: {len(clean)}')
print(f'Time span: {(clean.time[-1] - clean.time[0]).to_value("day"):.2f} d')


In [ ]:
ax = clean.scatter(s=2, title='Pi Mensae — TESS Sector 1')
ax.set_xlabel('Time [BTJD]')
ax.set_ylabel('Normalized flux')
plt.show()


## Cosa hai ottenuto
Una serie temporale fotometrica reale. **x** è il tempo, **y** è il flusso normalizzato della stella. Nel prossimo notebook costruiremo un periodogramma **Box Least Squares** e gli chiederemo di trovare da solo il periodo del transito.
